# MS MARCO RARS-v6 1M PQ-Specific Headroom

## Goal

Run the frozen, development-only 1M M32x8 headroom diagnostic at implementation commit `9abc24af7f2f8a6eb7a4d1416036c3151c51c924`. It separates IVF routing loss from PQ-specific Recall@100 loss and measures whether the remaining PQ flip signal is broad enough to justify implementing a new loss.

This notebook performs no adapter or encoder training, no PQ/codebook update, no RARS fitting, and no audit/future/external evaluation. A GO authorizes only a separately preregistered v6 loss implementation; it is not method success or SIGIR evidence.

## Measurement contract

For the already-observed 2,307-query `oracle_design` role, this notebook compares full 1M FP32 search, exact FP32 scoring inside the same 16 IVF probes, and the frozen CPU IVF-PQ search. All qrels stay in the Recall denominator. Only positives reachable through those probes may contribute PQ flip supervision; routing misses are never relabelled as PQ errors.

Unjudged boundary challengers are not treated as explicit non-relevant documents. The candidate union is scored with actual FP32 embeddings and actual Faiss PQ reconstructions before the fixed gate is applied.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EXPERIMENT_PYTHON = sys.executable
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q',
    'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
NUMPY_TARGET = Path('/content/rars-v6-numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
NUMPY_TARGET.mkdir(parents=True)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
    '--target', str(NUMPY_TARGET), 'numpy==1.26.4',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [
    str(NUMPY_TARGET), EXPERIMENT_ENV.get('PYTHONPATH', ''),
]))
EXPERIMENT_ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
installed_numpy_version = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__version__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version
installed_numpy_path = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__file__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert Path(installed_numpy_path).resolve().is_relative_to(NUMPY_TARGET.resolve())
print('Colab host-kernel NumPy (not used by experiments):',
      getattr(sys.modules.get('numpy'), '__version__', 'not-loaded'))
print('Fresh experiment-subprocess NumPy:', installed_numpy_version)

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
V3_IMPLEMENTATION_COMMIT = '05c2ae43b7d11783460822d10c590240dab1a399'
V6_IMPLEMENTATION_COMMIT = '9abc24af7f2f8a6eb7a4d1416036c3151c51c924'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
V3_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v3_oracle')
V6_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v6_headroom')

PARENT_WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
V3_WORK = Path('/content') / f'rars-v3-{V3_IMPLEMENTATION_COMMIT[:12]}'
for local_work in (PARENT_WORK, V3_WORK):
    if local_work.exists():
        shutil.rmtree(local_work)
    local_work.mkdir(parents=True)
PARENT_BUNDLES = PARENT_WORK / 'bundles'
PARENT_CANDIDATE_CACHE = PARENT_WORK / 'candidate-cache'
V3_BUNDLES = V3_WORK / 'bundles'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
OUTPUT = DRIVE / 'rars-v6-1m-headroom' / V6_IMPLEMENTATION_COMMIT[:12]

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_record(path, record):
    path = Path(path)
    assert path.is_file(), path
    assert path.stat().st_size == int(record['bytes']), path
    assert sha256_file(path) == record['sha256'], path

EXPERIMENT_PROBE = r'''
import json, os, sys
import faiss, numpy as np, torch
print(json.dumps({
    'python_version': '.'.join(map(str, sys.version_info[:3])),
    'python_full': sys.version,
    'numpy_version': np.__version__,
    'numpy_module_path': np.__file__,
    'torch_version': torch.__version__,
    'torch_cuda_version': str(torch.version.cuda),
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'faiss_version': str(getattr(faiss, '__version__', 'UNKNOWN')),
    'faiss_gpu_count': faiss.get_num_gpus(),
    'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
}, allow_nan=False))
'''

def probe_experiment_environment():
    return json.loads(subprocess.check_output(
        [EXPERIMENT_PYTHON, '-c', EXPERIMENT_PROBE],
        text=True, env=EXPERIMENT_ENV,
    ))

In [ ]:
def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(
        ['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True
    ).strip()
    dirty = subprocess.check_output(
        ['git', '-C', str(destination), 'status', '--porcelain'], text=True
    ).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(V3_REPO, V3_IMPLEMENTATION_COMMIT)
clone_exact(V6_REPO, V6_IMPLEMENTATION_COMMIT)

V3_PROTOCOL_PATH = V3_REPO / 'protocols/rars_v3_oracle_first_feasibility_v1.json'
V6_PROTOCOL_PATH = V6_REPO / 'protocols/rars_v6_1m_headroom_v1.json'
v3_protocol = json.loads(V3_PROTOCOL_PATH.read_text())
protocol = json.loads(V6_PROTOCOL_PATH.read_text())
assert protocol['status'] == 'FROZEN_BEFORE_FIRST_1M_HEADROOM_RUN'
assert protocol['method_revision_allowed'] is False
assert protocol['outcome_informed_revision_allowed'] is False
assert protocol['data_policy']['diagnostic_role']['role_id'] == 'oracle_design'
assert 'future_method_holdout' in protocol['data_policy']['forbidden_roles']
assert protocol['parent_lineage']['v3_implementation_commit'] == V3_IMPLEMENTATION_COMMIT

subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
], cwd=TRAIN_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v3_oracle_core.py',
    'tests/test_build_msmarco_rars_v3_oracle_bundles.py',
    'tests/test_rars_v3_oracle_protocol_contract.py',
], cwd=V3_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v6_headroom_core.py',
    'tests/test_evaluate_rars_v6_1m_headroom.py',
    'tests/test_rars_v6_headroom_protocol_contract.py',
], cwd=V6_REPO, check=True, env=EXPERIMENT_ENV)
print('Exact v6 implementation commit:', V6_IMPLEMENTATION_COMMIT)
print('Frozen v6 protocol SHA-256:', sha256_file(V6_PROTOCOL_PATH))

In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
minimum_free = protocol['resource_contract']['minimum_local_free_bytes_before_run']
assert shutil.disk_usage('/content').free >= minimum_free, 'Need 8 GB local disk'
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), (
    'The durable v6 output is non-empty. Do not delete or overwrite it; '
    'return its contents for audit before deciding how to continue.'
)

current_environment = probe_experiment_environment()
contract = protocol['execution_environment_contract']
assert current_environment['python_version'] == contract['python_version']
assert current_environment['numpy_version'] == contract['numpy_version']
assert Path(current_environment['numpy_module_path']).resolve().is_relative_to(
    NUMPY_TARGET.resolve()
)
assert current_environment['torch_version'] == contract['torch_version']
assert current_environment['torch_cuda_version'] == contract['torch_cuda_version']
assert current_environment['cuda_available'] is True
assert current_environment['faiss_gpu_count'] > 0
assert current_environment['faiss_version'] == '1.12.0'
assert contract['gpu_name_must_contain'] in current_environment['gpu_name']
assert current_environment['cublas_workspace_config'] == contract['cublas_workspace_config']
print(json.dumps(current_environment, indent=2))

## Reproduce the registered design identity

The next cells rematerialize the exact v2.2 parent bundle and the qrels-free v3 role split at their pinned commits. The historical parent builder necessarily reads the shared qrels cache. V6 later selects positive judgments only for the registered design query IDs.

No v3 role labels are materialized. The audit role is never passed to v6, and the future role must remain identity-only.

In [ ]:
builder = [
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(PARENT_CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(PARENT_BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
bundle_summary = json.loads((PARENT_BUNDLES / 'bundle_build_summary.json').read_text())
assert bundle_summary['outer_validation_built'] is False
assert set(bundle_summary['roles']) == {'inner_train', 'inner_validation'}

subprocess.run([
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(PARENT_BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)

parent = v3_protocol['parent_lineage']
parent_hashes = {
    'parent_inner_train_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/v2_2_manifest.json'),
    'parent_inner_train_source_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/manifest.json'),
    'parent_inner_train_query_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/query_manifest.json'),
    'closed_inner_validation_query_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_validation/query_manifest.json'),
    'parent_v2_2_split_audit_sha256': sha256_file(PARENT_BUNDLES / 'v2_2_split_audit.json'),
}
for key, actual in parent_hashes.items():
    assert actual == parent[key], (key, actual, parent[key])
print('Exact v2.2 parent rematerialized.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/build_msmarco_rars_v3_oracle_bundles.py'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--output-root', str(V3_BUNDLES),
    '--protocol', str(V3_PROTOCOL_PATH),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--n-docs', '1000000',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
candidate_summary = json.loads(
    (V3_BUNDLES / 'v3_oracle_bundle_freeze_summary.json').read_text()
)
assert candidate_summary['status'] == 'V3_QRELS_FREE_CANDIDATE_BUNDLES_FROZEN'
assert candidate_summary['parent_label_payload_bytes_read'] is False
assert candidate_summary['qrels_opened_or_parsed'] is False

ROLE_LABEL_FILES = {
    'candidate_relevance.uint8.npy',
    'relevant_counts.int32.npy',
    'v3_role_labels_started.json',
    'v3_role_labels_manifest.json',
}
for role in ('oracle_design', 'oracle_audit'):
    assert not ROLE_LABEL_FILES.intersection(
        path.name for path in (V3_BUNDLES / role).iterdir()
    )
future_files = {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()}
assert future_files == {'query_manifest.json', 'v3_identity_manifest.json'}
print('Qrels-free design identity frozen; audit unlabeled; future identity-only.')

## Run the no-training 1M diagnostic

This is the first v6 metric access. It loads the 1M FP16 corpus into a temporary FP32 T4 tensor, runs full exact and same-probe exact retrieval, performs the frozen CPU IVF-PQ search, scores the PQ flip union, and writes the formal gate. Do not interrupt this cell or edit the durable output directory.

The evaluator records stage wall time, peak host/CUDA memory, and disk headroom. A resource failure is `STOP_RESOURCE_SMOKE_FAILED`, not evidence for or against the method idea.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V6_REPO / 'scripts/evaluate_rars_v6_1m_headroom.py'),
    '--design-role-dir', str(V3_BUNDLES / 'oracle_design'),
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--index', str(INDEX),
    '--output-dir', str(OUTPUT),
    '--protocol', str(V6_PROTOCOL_PATH),
    '--source-commit', V6_IMPLEMENTATION_COMMIT,
    '--scratch-dir', '/content',
    '--use-gpu',
], check=True, cwd=V6_REPO, env=EXPERIMENT_ENV)
print('RARS-v6 1M headroom diagnostic completed.')

In [ ]:
complete_path = OUTPUT / 'headroom_complete.json'
result_path = OUTPUT / 'headroom_result.json'
complete = json.loads(complete_path.read_text())
result = json.loads(result_path.read_text())
assert complete['status'] == 'RARS_V6_1M_HEADROOM_COMPLETE'
assert result['status'] == 'RARS_V6_1M_HEADROOM_COMPLETE'
assert complete['source_commit'] == V6_IMPLEMENTATION_COMMIT
assert result['source_commit'] == V6_IMPLEMENTATION_COMMIT
assert complete['formal_decision'] == result['formal_decision']
assert complete['formal_decision'] in {
    'GO_TO_V6_LOSS_IMPLEMENTATION',
    'STOP_NO_DISTRIBUTED_PQ_HEADROOM',
}
assert complete['training_performed'] is False
assert complete['adapter_used'] is False
assert complete['rars_used'] is False
assert complete['future_or_audit_role_opened'] is False
assert result['signal_gate']['training_authorized'] is False
verify_record(result_path, complete['result'])
for filename, record in complete['outputs'].items():
    verify_record(OUTPUT / filename, record)
missing_outputs = [
    filename for filename in protocol['required_outputs']
    if not (OUTPUT / filename).is_file()
]
assert not missing_outputs, missing_outputs
report = {
    'formal_decision': result['formal_decision'],
    'mean_recall': result['mean_recall'],
    'recall_gap_decomposition': result['recall_gap_decomposition'],
    'uncapped_flip_support': result['flip_support']['uncapped'],
    'capped_flip_support': result['flip_support']['capped'],
    'flip_candidate_union': result['flip_candidate_union'],
    'failed_gates': result['signal_gate']['failed_gates'],
    'telemetry': result['telemetry'],
    'result_sha256': sha256_file(result_path),
}
print(json.dumps(report, indent=2, allow_nan=False))

## Checks and next steps

Interpret the formal decision literally:

- `STOP_NO_DISTRIBUTED_PQ_HEADROOM`: at least one quantity, breadth, ESS, concentration, or corpus-coverage gate failed. Stop M32x8 loss development in this setting; do not tune the gate after seeing the result.
- `GO_TO_V6_LOSS_IMPLEMENTATION`: freeze a separate training protocol for Top-10 protection and Top-100 repair. This diagnostic still does not authorize training, RARS fitting, audit/future access, or a paper claim.

Return the printed report plus `headroom_result.json` and `headroom_complete.json` for audit. Do not rerun into the same durable directory.